In [1]:
from matplotlib import pyplot as plt
import h5py
import numpy as np

In [2]:
dbm_colors = {
    0.5: 'tab:blue',
    1: 'tab:orange',
    4: 'tab:green',
    7: 'tab:red', 
    14: 'tab:purple',
}

In [3]:
def collect_fitting_factor_results(
    result_file
):
    results_dict = {}
    with h5py.File(result_file, 'r') as result_f:
        for signal_number_str in result_f.keys():
            signal_number  = int(signal_number_str)
            sig_grp = result_f[signal_number_str]
            results_dict[signal_number] = {
                'fitting_factor': sig_grp['fitting_factor'][:],
                'sample_times': sig_grp['sample_times'][:]
            }
    return results_dict

In [ ]:
fitting_factor_results = {
    days_before: collect_fitting_factor_results(
        f'results/{days_before}/fitting_factor_{days_before}.hdf'
    ) for days_before in
    [0.5, 1, 4, 7, 14]
}

# Inverting the keys:
ff_results = {}
for a, inner in fitting_factor_results.items():
    for b, value in inner.items():
        if b not in ff_results:
            ff_results[b] = {}
        print(value)
        ff_results[b][a] = value / max(value)

TypeError: unsupported operand type(s) for /: 'dict' and 'str'

In [ ]:
def plot_optimal_snr_fitting_factor(
    cutoff_days,
    optimal_snr_data,
    fitting_factor_results,
    yscale='linear',
    xscale='linear',
    signal_number=None
):
    fig, opt_ax = plt.subplots()

    fig.suptitle(f"Signal {signal_number} Optimal SNR and fitting factor over time")

    opt_ax.plot(cutoff_days, optimal_snr_data, c='k')
    ff_ax = opt_ax.twinx()
    for days_before_merger, ff_result in fitting_factor_results.items():
        if days_before_merger < 3:
            continue
        ff_ax.plot(
            ff_result['sample_times'],
            ff_result['fitting_factor'],
            c=dbm_colors[float(days_before_merger)],
            linestyle=':'
        )

        ff_interp = np.interp(
            cutoff_days,
            ff_result['sample_times'],
            ff_result['fitting_factor']
        )
        product = ff_interp * optimal_snr_data
        opt_ax.plot(
            cutoff_days,
            product,
            c=dbm_colors[days_before_merger],
            linestyle='--'
        )

        opt_ax.plot(
            [],[],
            c=dbm_colors[days_before_merger],
            label=days_before_merger
        )

    leg1 = opt_ax.legend(loc='upper left', title='Days before merger')

    lines2 = [
        opt_ax.plot([],[], c='k', linestyle='-')[0],
        opt_ax.plot([],[], c='k', linestyle=':')[0],
        opt_ax.plot([],[], c='k', linestyle='--')[0],
    ]
    lbls2 = [
        "Optimal SNR",
        "Fitting Factor",
        "Product",
    ]

    opt_ax.legend( lines2, lbls2,loc='lower left',)
    opt_ax.add_artist(leg1)

    ff_ax.set_yscale(yscale)
    opt_ax.set_xscale(xscale)
    opt_ax.set_yscale(yscale)
    opt_ax.grid()
    opt_ax.set_ylabel('SNR')
    opt_ax.set_xlim([-20, 5])
    ff_ax.set_ylabel('Fitting Factor')
    ff_ax.set_ylim(bottom=1e-2)
    opt_ax.set_ylim(bottom=1e-2)
    opt_ax.set_xlabel('Time')


In [ ]:
optimal_snr_over_time = {}
for signal_number in range(15):
    with h5py.File(f'results/optimal_snr_over_time_{signal_number}.hdf','r') as result_f:
        cutoff_days = result_f['cutoff_days'][:]
        optimal_snr_over_time[signal_number] = result_f[f'optimal_snr_signal_{signal_number}'][:]


In [ ]:

for signal_number in np.arange(15):
    if signal_number not in ff_results:
        continue
    if signal_number not in optimal_snr_over_time:
        continue
    plot_optimal_snr_fitting_factor(
        cutoff_days,
        optimal_snr_over_time[signal_number],
        fitting_factor_results=ff_results[signal_number],
        signal_number=signal_number,
        yscale='log',
        xscale='linear'
    )